In [144]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [250]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
from tqdm import tqdm
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
#warnings.filterwarnings('ignore')

In [251]:
# Time
start = dt.datetime(2019,6,18)
end = dt.datetime(2019,6,20)
print(start,end)

2019-06-18 00:00:00 2019-06-20 00:00:00


In [252]:
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$gte': start,'$lt': end}},{"sign_up_details":1, "created_at":1}): # end condition
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
users = pd.DataFrame(dic_flattened)
users = users[["_id","created_at","sign_up_details_device_id"]]
users.columns = ["user_id","create_time","device_id"]
len(users)
#users = users[users['user_id']!='5cfddaf7bba72a0018ea594f']

533

In [253]:
users.sort_values(['device_id','create_time'],ascending=False,inplace=True)
users.drop_duplicates('device_id',inplace=True)
print(len(users))
users.head()

529


,user_id,create_time,device_id
471,5d0a554c3c8b7e0018c755e1,2019-06-19 15:31:24.753,feb41861765bf689823278c636b38288
36,5d08704ec61219034a7a854a,2019-06-18 05:02:06.977,fe9d3fcffddfd7d1d2af1c43105570de
388,5d0a14b166bb01001125e7ba,2019-06-19 10:55:45.225,fde75c6b8b8e605ef0b2308391b18255
66,5d088f14a6264f0359c3a43d,2019-06-18 07:13:24.720,fda9cf55c0a5a9f443c4b05d17b46607
292,5d09b5b666bb01001119841f,2019-06-19 04:10:30.497,fd35c951d552c73e2d3d10e9eb0df476


In [254]:
team_cursor = cursor.superstars.teams
aw_team = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1,'created_at':1,'name':1}): 
    aw_team.append(documents)
dic_flattened = [flatten(d) for d in aw_team]
teams = pd.DataFrame(dic_flattened)
teams = teams[teams["user"].isin(users["user_id"])]
teams = teams[["_id","user",'created_at','name']]
teams.columns = ["team_id", "user_id","team_create_time","team_name"]

In [255]:
teams = teams[teams['team_name']!='sexy daddies']
print(len(teams))
teams.head()

529


,team_id,user_id,team_create_time,team_name
0,5d083744a6264f0359b27670,5d083744a6264f0359b27664,2019-06-18 00:58:44.439,Smashing Panthers
1,5d083866a6264f0359b27b6d,5d083866a6264f0359b27b61,2019-06-18 01:03:34.557,Amazing Bull
2,5d0839f7c61219034a70b4d5,5d0839f7c61219034a70b4c9,2019-06-18 01:10:15.430,Smashing Warriorst
3,5d083cccc61219034a710ea3,5d083cccc61219034a710e97,2019-06-18 01:22:20.351,Fiery Rhinos
4,5d083fd4a6264f0359b3873c,5d083fd4a6264f0359b38730,2019-06-18 01:35:16.983,Falcon Thunders भुपेन बना सा


In [256]:
player_cursor = cursor.superstars.players
aw_players = []
for documents in player_cursor.find({'created_at': { '$gte': start}},{'team':1,'level':1}):
    aw_players.append(documents)
dic_flattened = [flatten(d) for d in aw_players]
players = pd.DataFrame(dic_flattened)
players = players[players["team"].isin(teams["team_id"])]
players = players[["_id","team",'level']]
players.columns = ["player_id", "team_id",'player_level']

In [257]:
print(len(players))
players.head()

1329


,player_id,team_id,player_level
0,5d083744a6264f0359b27674,5d083744a6264f0359b27670,1
1,5d083744a6264f0359b27676,5d083744a6264f0359b27670,2
2,5d083866a6264f0359b27b71,5d083866a6264f0359b27b6d,9
3,5d083866a6264f0359b27b73,5d083866a6264f0359b27b6d,5
4,5d0839f7c61219034a70b4d9,5d0839f7c61219034a70b4d5,1


In [258]:
users_team_player = pd.merge(teams,players,on='team_id')
print(len(users_team_player))
users_team_player.head()

1329


,team_id,user_id,team_create_time,team_name,player_id,player_level
0,5d083744a6264f0359b27670,5d083744a6264f0359b27664,2019-06-18 00:58:44.439,Smashing Panthers,5d083744a6264f0359b27674,1
1,5d083744a6264f0359b27670,5d083744a6264f0359b27664,2019-06-18 00:58:44.439,Smashing Panthers,5d083744a6264f0359b27676,2
2,5d083866a6264f0359b27b6d,5d083866a6264f0359b27b61,2019-06-18 01:03:34.557,Amazing Bull,5d083866a6264f0359b27b71,9
3,5d083866a6264f0359b27b6d,5d083866a6264f0359b27b61,2019-06-18 01:03:34.557,Amazing Bull,5d083866a6264f0359b27b73,5
4,5d083866a6264f0359b27b6d,5d083866a6264f0359b27b61,2019-06-18 01:03:34.557,Amazing Bull,5d083b24c61219034a70c29c,4


In [259]:
skill_cursor = cursor.superstars.player_skill_logs
aw_skill = []
for documents in skill_cursor.aggregate([{'$unwind':"$details"}, 
                                    {"$match" : {'created_at': {'$gte': start},"details.type" : 'TRAINING_PROGRESS'}}]):
    aw_skill.append(documents)
dic_flattened = [flatten(d) for d in aw_skill]
players_trained = pd.DataFrame(dic_flattened)
players_trained = players_trained[players_trained["player"].isin(players["player_id"])]
players_trained = players_trained[["created_at","details__id","player",'details_to_value']]
players_trained.columns = ["event_timestamp","training_id","player_id",'value']

In [268]:
aw_skill[-1]

{'_id': ObjectId('5d132d4f7119e60013995687'),
 'player': ObjectId('5d132d227119e60013993979'),
 'details': {'_id': ObjectId('5d132e117119e60013997e50'),
  'from_value': '0',
  'to_value': '1200',
  'type': 'TRAINING_PROGRESS'},
 'created_at': datetime.datetime(2019, 6, 26, 8, 31, 11, 985000),
 'updated_at': datetime.datetime(2019, 6, 26, 8, 34, 25, 519000),
 '__v': 5}

In [267]:
print(len(players_trained))
players_trained.sort_values(['event_timestamp'],inplace=True)
players_trained.head()

2034


,event_timestamp,training_id,player_id,value
42,2019-06-18 01:13:28.667,5d083ab8a6264f0359b2b02d,5d083866a6264f0359b27b71,200
43,2019-06-18 01:13:28.667,5d083ac6a6264f0359b2b07a,5d083866a6264f0359b27b71,400
44,2019-06-18 01:13:28.667,5d083ad0c61219034a70bb91,5d083866a6264f0359b27b71,600
45,2019-06-18 01:13:28.667,5d083ad8a6264f0359b2b0a9,5d083866a6264f0359b27b71,800
46,2019-06-18 01:13:28.667,5d083ae0a6264f0359b2ca17,5d083866a6264f0359b27b71,1000


In [261]:
users_team_player_trained = pd.merge(users_team_player[['user_id','player_id']],
                                     players_trained[['event_timestamp','player_id','value']], on='player_id')

In [262]:
users_team_player_trained.drop('player_id',inplace=True,axis=1)
users_team_player_trained['activity'] = 'training'
users_team_player_trained['value'] = users_team_player_trained['value'].astype(int)

In [265]:
print(len(users_team_player_trained))
users_team_player_trained.sort_values(['user_id','event_timestamp'],inplace=True)
users_team_player_trained

2034


,user_id,event_timestamp,value,activity
0,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,200,training
1,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,400,training
2,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,600,training
3,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,800,training
4,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,1000,training
5,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,1200,training
6,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,1400,training
7,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,1600,training
8,5d083866a6264f0359b27b61,2019-06-18 01:14:23.624,400,training
9,5d083866a6264f0359b27b61,2019-06-18 01:14:23.624,600,training


In [193]:
train_time = pd.read_csv('Training_Centre_Sheet2.csv')

In [194]:
train_time = train_time.iloc[:-2]

In [195]:
train_time = train_time[['next_level','Seconds.1']]
train_time.columns = ['value','minutes']
train_time['value'] = (train_time['value']).astype(int)

In [196]:
train_time.head()

,value,minutes
0,200,0.17
1,400,0.5
2,600,1
3,800,5
4,1000,15


In [201]:
users_team_player_trained = pd.merge(users_team_player_trained,train_time, on='value')
users_team_player_trained = users_team_player_trained[['user_id','event_timestamp','minutes','activity']]

In [202]:
users_team_player_trained.head()

,user_id,event_timestamp,minutes,activity
0,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,0.17,training
1,5d083866a6264f0359b27b61,2019-06-18 01:16:05.077,0.17,training
2,5d083866a6264f0359b27b61,2019-06-18 01:22:38.214,0.17,training
3,5d083866a6264f0359b27b61,2019-06-19 15:27:41.052,0.17,training
4,5d08427dc61219034a7260f1,2019-06-18 04:10:34.959,0.17,training


In [235]:
c_cards = cursor.superstars.user_collectables_logs # this query will give me how many speedup_cards have been used. 
                                                    #Each tap will be counted as one use of the speedup card
aw_cards = []
for documents in c_cards.aggregate([{'$unwind':"$data"}, 
                                    {"$match" : {'data.quantity': {'$lt': 0},
                                                 'type':'TRAINING_SPEEDUP_CARD',
                                                 'data.reason_type':'SPEEDUP_USE',
                                                 'data.created_at': {'$gte': start}}}]):
    aw_cards.append(documents)
 
dic_flattened = [flatten(d) for d in aw_cards]
cards = pd.DataFrame(dic_flattened)
cards = cards[cards["user"].isin(users_team_player_trained['user_id'])]   # total_trained is the number of times each user has trained
cards = cards[['user',"data_created_at",'data_catalogue_id']]
cards.columns = ['user_id',"event_timestamp",'value']

In [236]:
cards['value'] = cards['value'].astype(int)
cards.head()

,user_id,event_timestamp,value
4417,5d083866a6264f0359b27b61,2019-06-18 01:22:53.898,1
4418,5d083866a6264f0359b27b61,2019-06-18 01:33:05.124,1
4419,5d083866a6264f0359b27b61,2019-06-19 08:08:20.117,1
4420,5d083866a6264f0359b27b61,2019-06-19 08:08:24.379,1
4421,5d083866a6264f0359b27b61,2019-06-19 08:08:24.721,1


In [249]:
cards['activity'] = 'speed_up'
cards.sort_values(['user_id','event_timestamp'],inplace=True)
cards.head(5)

,user_id,event_timestamp,minutes,activity
0,5d083866a6264f0359b27b61,2019-06-18 01:22:53.898,1 MIN SPEED UP,speed_up
1,5d083866a6264f0359b27b61,2019-06-18 01:33:05.124,1 MIN SPEED UP,speed_up
2,5d083866a6264f0359b27b61,2019-06-19 08:08:20.117,1 MIN SPEED UP,speed_up
3,5d083866a6264f0359b27b61,2019-06-19 08:08:24.379,1 MIN SPEED UP,speed_up
4,5d083866a6264f0359b27b61,2019-06-19 08:08:24.721,1 MIN SPEED UP,speed_up


In [238]:
exports = [
  {
    "id": 1,
    "speedup_time": (1 * 1 * 60),
    "hitcoins_needed": 1,
    "title": "1 MIN SPEED UP",
    "description": "Reduces training time by 1 min"
  },
  {
    "id": 2,
    "speedup_time": (1 * 5 * 60),
    "hitcoins_needed": 3,
    "title": "5 MIN SPEED UP",
    "description": "Reduces training time by 5 min"
  },
  {
    "id": 3,
    "speedup_time": (1 * 15 * 60),
    "hitcoins_needed": 7,
    "title": "15 MIN SPEED UP",
    "description": "Reduces training time by 15 min"
  },
  {
    "id": 4,
    "speedup_time": (1 * 60 * 60),
    "hitcoins_needed": 22,
    "title": "1 HOUR SPEED UP",
    "description": "Reduces training time by 1 hour"
  },
  {
    "id": 5,
    "speedup_time": (3 * 60 * 60),
    "hitcoins_needed": 50,
    "title": "3 HOUR SPEED UP",
    "description": "Reduces training time by 3 hour"
  },
  {
    "id": 6,
    "speedup_time": (8 * 60 * 60),
    "hitcoins_needed": 100,
    "title": "8 HOUR SPEED UP",
    "description": "Reduces training time by 8 hour"
  }
]

In [239]:
chinki = pd.DataFrame(exports)

In [240]:
chinki = chinki[['id','title']]

In [241]:
cards = pd.merge(cards, chinki, right_on = 'id',left_on='value')
cards = cards[['user_id','event_timestamp','title','activity']]
cards.columns = ['user_id','event_timestamp','minutes','activity']

In [242]:
cards.head()

,user_id,event_timestamp,minutes,activity
0,5d083866a6264f0359b27b61,2019-06-18 01:22:53.898,1 MIN SPEED UP,speed_up
1,5d083866a6264f0359b27b61,2019-06-18 01:33:05.124,1 MIN SPEED UP,speed_up
2,5d083866a6264f0359b27b61,2019-06-19 08:08:20.117,1 MIN SPEED UP,speed_up
3,5d083866a6264f0359b27b61,2019-06-19 08:08:24.379,1 MIN SPEED UP,speed_up
4,5d083866a6264f0359b27b61,2019-06-19 08:08:24.721,1 MIN SPEED UP,speed_up


In [243]:
print(len(cards)+len(users_team_player_trained))

3267


In [244]:
training_with_speedup = pd.concat([cards, users_team_player_trained],ignore_index=True)
len(training_with_speedup)

3267

In [245]:
training_with_speedup.sort_values('event_timestamp',inplace=True)

In [248]:
training_with_speedup

,user_id,event_timestamp,minutes,activity
1233,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,0.17,training
1622,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,0.5,training
3130,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,120,training
2083,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,1,training
3046,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,60,training
2452,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,5,training
2728,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,15,training
2918,5d083866a6264f0359b27b61,2019-06-18 01:13:28.667,30,training
1623,5d083866a6264f0359b27b61,2019-06-18 01:14:23.624,0.5,training
2084,5d083866a6264f0359b27b61,2019-06-18 01:14:23.624,1,training
